# Meme Vector Index Builder - GPU Accelerated

This notebook builds the FAISS vector index for semantic meme search using GPU acceleration.

**Important:** Make sure GPU is enabled:
- Go to: Runtime → Change runtime type → Hardware accelerator → GPU

**Estimated Time:** 2-4 hours with GPU (vs 24-30 hours on CPU)

## Step 1: Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("WARNING: GPU not available! Please enable GPU in Runtime settings.")

## Step 2: Install Dependencies

In [ ]:
!pip install sentence-transformers faiss-cpu scikit-learn tqdm --quiet
print("Dependencies installed!")

## Step 3: Upload Project Files

Upload your project as a zip file or clone from GitHub.

In [ ]:
from google.colab import files
import zipfile
import os

print("Please upload your project zip file...")
print("The zip should contain:")
print("  - meme_agent/")
print("  - ImgFlip575K_Dataset/")
print("  - storage/")
print("")

uploaded = files.upload()
filename = list(uploaded.keys())[0]

print(f"\nExtracting {filename}...")
with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall('.')

print("Extraction complete!")
print("\nProject structure:")
!ls -la

## Step 4: Verify Dataset

In [ ]:
import os

memes_dir = "ImgFlip575K_Dataset/dataset/memes"
if os.path.exists(memes_dir):
    meme_files = [f for f in os.listdir(memes_dir) if f.endswith('.json')]
    print(f"Found {len(meme_files)} meme template files")
    
    # Count total memes
    import json
    total_memes = 0
    for f in meme_files[:10]:  # Sample first 10
        with open(os.path.join(memes_dir, f), 'r') as fp:
            data = json.load(fp)
            total_memes += len(data)
    print(f"Sample count from first 10 files: {total_memes} memes")
    print(f"Estimated total: ~{total_memes * len(meme_files) // 10} memes")
else:
    print(f"ERROR: Dataset not found at {memes_dir}")
    print("Make sure ImgFlip575K_Dataset is in your uploaded zip!")

## Step 5: Create Optimized Build Script for GPU

In [ ]:
%%writefile build_index_gpu.py
"""
GPU-Optimized Vector Index Builder for Meme Dataset
Runs on Google Colab with GPU acceleration
"""

import os
import sys
import json
import time
from pathlib import Path
from typing import List, Tuple, Dict, Any
import numpy as np
from tqdm import tqdm

# Configuration
BATCH_SIZE = 128  # Larger batch size for GPU
MODEL_NAME = "all-MiniLM-L6-v2"
STORAGE_DIR = "storage/vector_index"
MEMES_DIR = "ImgFlip575K_Dataset/dataset/memes"

def load_meme_data(memes_dir: str) -> Tuple[List[str], List[str], List[Dict[str, Any]]]:
    """Load all meme data from JSON files."""
    texts = []
    ids = []
    meme_data_list = []
    
    if not os.path.exists(memes_dir):
        raise FileNotFoundError(f"Memes directory not found: {memes_dir}")
    
    json_files = sorted([f for f in os.listdir(memes_dir) if f.endswith('.json')])
    total_files = len(json_files)
    
    print(f"Loading memes from {total_files} template files...")
    
    for filename in tqdm(json_files, desc="Loading memes"):
        template_id = os.path.splitext(filename)[0]
        filepath = os.path.join(memes_dir, filename)
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                memes = json.load(f)
            
            if not isinstance(memes, list):
                continue
            
            for meme_idx, meme in enumerate(memes):
                boxes = meme.get('boxes', [])
                if not boxes:
                    continue
                
                caption = ' '.join(str(box) for box in boxes)
                title = meme.get('metadata', {}).get('title', '')
                
                if title and title.lower() not in caption.lower():
                    search_text = f"{caption} {title}"
                else:
                    search_text = caption
                
                meme_id = f"{template_id}_{meme_idx}"
                
                texts.append(search_text.strip())
                ids.append(meme_id)
                meme_data_list.append({
                    'template_id': template_id,
                    'meme_idx': meme_idx,
                    'caption': caption,
                    'title': title,
                    'url': meme.get('url', ''),
                    'post_url': meme.get('post', '')
                })
                
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            continue
    
    return texts, ids, meme_data_list


def generate_embeddings_gpu(texts: List[str], batch_size: int = 128) -> np.ndarray:
    """Generate embeddings using GPU-accelerated sentence-transformers."""
    from sentence_transformers import SentenceTransformer
    
    print(f"Loading model: {MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME, device='cuda')
    print(f"Model loaded on GPU: {model.device}")
    
    print(f"\nGenerating embeddings for {len(texts)} memes...")
    print(f"Batch size: {batch_size}")
    
    # Generate embeddings with progress bar
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    print(f"Embeddings shape: {embeddings.shape}")
    return embeddings


def build_faiss_index(embeddings: np.ndarray, ids: List[str], storage_dir: str):
    """Build and save FAISS index."""
    import faiss
    
    print("\nBuilding FAISS index...")
    
    # Ensure embeddings are float32
    embeddings = embeddings.astype(np.float32)
    
    # Create index (Inner Product = Cosine Similarity for normalized vectors)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    
    # Create storage directory
    os.makedirs(storage_dir, exist_ok=True)
    
    # Save index
    index_path = os.path.join(storage_dir, "index.faiss")
    print(f"Saving index to {index_path}...")
    faiss.write_index(index, index_path)
    
    # Save ID mapping
    id_map = {i: id_ for i, id_ in enumerate(ids)}
    id_map_path = os.path.join(storage_dir, "id_map.json")
    print(f"Saving ID map to {id_map_path}...")
    with open(id_map_path, 'w', encoding='utf-8') as f:
        json.dump(id_map, f)
    
    # Save metadata
    metadata = {
        'total_memes': len(ids),
        'embedding_dim': dimension,
        'model_name': MODEL_NAME,
        'built_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'built_on': 'Google Colab GPU'
    }
    metadata_path = os.path.join(storage_dir, "metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Save meme data lookup
    print("Index built successfully!")
    return index


def main():
    print("="*60)
    print("Meme Vector Index Builder - GPU Accelerated")
    print("="*60)
    
    start_time = time.time()
    
    # Load data
    texts, ids, meme_data = load_meme_data(MEMES_DIR)
    print(f"\nLoaded {len(texts)} memes")
    
    # Generate embeddings
    embeddings = generate_embeddings_gpu(texts, batch_size=BATCH_SIZE)
    
    # Build index
    build_faiss_index(embeddings, ids, STORAGE_DIR)
    
    # Save meme data
    meme_data_path = os.path.join(STORAGE_DIR, "meme_data.json")
    print(f"Saving meme data to {meme_data_path}...")
    with open(meme_data_path, 'w', encoding='utf-8') as f:
        json.dump(meme_data, f)
    
    elapsed = time.time() - start_time
    hours, remainder = divmod(int(elapsed), 3600)
    minutes, seconds = divmod(remainder, 60)
    
    print("\n" + "="*60)
    print("BUILD COMPLETE!")
    print("="*60)
    print(f"Total time: {hours}h {minutes}m {seconds}s")
    print(f"\nOutput files in {STORAGE_DIR}/:")
    for f in os.listdir(STORAGE_DIR):
        size = os.path.getsize(os.path.join(STORAGE_DIR, f))
        print(f"  - {f}: {size / (1024*1024):.1f} MB")


if __name__ == "__main__":
    main()

## Step 6: Run the Index Build

This will take 2-4 hours with GPU. The progress bar will show the embedding generation progress.

In [ ]:
import time
start = time.time()

# Run the build script
!python build_index_gpu.py

elapsed = time.time() - start
print(f"\n\nTotal execution time: {elapsed/3600:.2f} hours")

## Step 7: Verify Output Files

In [ ]:
import os

storage_dir = "storage/vector_index"
required_files = ["index.faiss", "id_map.json", "metadata.json", "meme_data.json"]

print("Checking output files...")
print("="*50)

all_exist = True
for filename in required_files:
    filepath = os.path.join(storage_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        if size > 1024*1024:  # > 1MB
            print(f"✓ {filename}: {size/(1024*1024):.1f} MB")
        else:
            print(f"✓ {filename}: {size/1024:.1f} KB")
    else:
        print(f"✗ {filename}: MISSING")
        all_exist = False

print("="*50)
if all_exist:
    print("All required files exist!")
    
    # Show metadata
    with open(os.path.join(storage_dir, "metadata.json"), 'r') as f:
        metadata = json.load(f)
    print("\nIndex Metadata:")
    for key, value in metadata.items():
        print(f"  {key}: {value}")
else:
    print("Some files are missing! Build may have failed.")

## Step 8: Download the Vector Index

Download the complete vector_index folder as a zip file.

In [ ]:
import shutil
from google.colab import files

print("Creating zip file for download...")

# Create zip of the vector_index folder
zip_path = shutil.make_archive('vector_index', 'zip', 'storage/vector_index')

print(f"Zip file created: {zip_path}")
print(f"File size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")
print("\nDownload should start automatically...")
print("\nAfter download:")
print("1. Extract the zip file")
print("2. Replace contents of your local storage/vector_index/ folder")
print("3. Run: python meme_agent/scripts/test_vector_search.py")

files.download(zip_path)

## Step 9: Quick Test (Optional)

Test the index before downloading.

In [ ]:
# Quick test of the built index
import sys
sys.path.insert(0, '.')

from meme_agent.embeddings import generate_query_embedding
from meme_agent.vector_store import VectorStore

# Load the index
store = VectorStore(
    index_path="storage/vector_index/index.faiss",
    id_map_path="storage/vector_index/id_map.json"
)

if store.load_index():
    print(f"Index loaded successfully!")
    print(f"Total vectors: {store.size()}")
    
    # Test search
    test_queries = ["exam stress", "monday mood", "happy birthday"]
    
    for query in test_queries:
        query_emb = generate_query_embedding(query)
        ids, scores = store.search(query_emb, top_k=3)
        
        print(f"\nQuery: '{query}'")
        for id_, score in zip(ids, scores):
            print(f"  {id_}: {score:.4f}")
else:
    print("Failed to load index!")